# TeClass zero-shot LLM baseline (Sarvam)

Zero-shot classification on TeClass using Sarvam AI's India-focused LLM. Compared against the paper's fine-tuned baselines, without any training.

Model: sarvam-m (Indian languages specialist, free tier).

Prompt: zero-shot (task definition only).

Metrics: per-class F1 for HREL, MREL, LREL plus weighted and macro F1.

## 1. Setup

In [ ]:
!pip install -q requests tqdm

In [ ]:
import os
import re
import json
import time
import random
from pathlib import Path

import pandas as pd
import numpy as np
import requests
from tqdm.auto import tqdm

from sklearn.metrics import f1_score, confusion_matrix

try:
    from google.colab import userdata
    SARVAM_KEY = userdata.get('SARVAM_API')
except Exception:
    SARVAM_KEY = os.environ.get('SARVAM_API', 'PASTE_SARVAM_KEY_HERE')

SARVAM_URL = "https://api.sarvam.ai/v1/chat/completions"

r = requests.post(
    SARVAM_URL,
    headers={
        "api-subscription-key": SARVAM_KEY,
        "Content-Type": "application/json",
    },
    json={
        "model": "sarvam-105b-conversations",
        "messages": [{"role": "user", "content": "Say ok."}],
        "temperature": 0,
        "max_tokens": 50,
    },
    timeout=30,
)
print("status:", r.status_code)
if r.status_code == 200:
    print("sarvam-m:", r.json()["choices"][0]["message"]["content"].strip())
else:
    print("error:", r.text[:300])

status: 200
sarvam-m: Ok.


## 2. Load TeClass

In [ ]:
def load(path):
    df = pd.read_csv(path)
    cols = {c.lower(): c for c in df.columns}
    art  = next(cols[c] for c in ["article", "text", "body", "news_article"] if c in cols)
    head = next(cols[c] for c in ["headline", "title", "news_headline"] if c in cols)
    lab  = next(cols[c] for c in ["label", "category", "class"] if c in cols)
    df = df[[art, head, lab]].rename(columns={art: "article", head: "headline", lab: "label"})
    df["label"] = df["label"].astype(str).str.upper().str.strip()
    df = df[df["label"].isin(["HREL", "MREL", "LREL"])].reset_index(drop=True)
    return df

test_df = load("test.csv")

LABELS = ["HREL", "MREL", "LREL"]
lab2id = {"HREL": 0, "MREL": 1, "LREL": 2}

print("test:", test_df.shape)
print(test_df["label"].value_counts())

test: (3927, 3)
label
MREL    1523
HREL    1278
LREL    1126
Name: count, dtype: int64


## 3. Zero-shot prompt

In [ ]:
DEFS = """IMPORTANT: Classes are approximately balanced in the test set
(roughly 1/3 HREL, 1/3 MREL, 1/3 LREL). Do NOT default to HREL.
Look carefully for LREL cues like sensational language or claims unsupported by the article.

Categories:
- HREL: headline is a faithful summary of the article's main event.
- MREL: headline is loosely related, covers a side event or weak inference.
- LREL: headline is misleading, sensational, clickbait, or draws unsupported conclusions."""

def cut(text, n=2200):
    text = str(text)
    return text if len(text) <= n else text[:n] + " [...]"

def p_zeroshot(a, h):
    return (f"Classify the relevance between this Telugu news article and headline.\n\n"
            f"{DEFS}\n\nArticle: {cut(a)}\n\nHeadline: {h}\n\n"
            f"Reply with ONE label only: HREL, MREL, or LREL.")

PROMPTS = {"zeroshot": p_zeroshot}

print(p_zeroshot(test_df.iloc[0]["article"], test_df.iloc[0]["headline"])[:500], "...")

Classify the relevance between this Telugu news article and headline.

IMPORTANT: Classes are approximately balanced in the test set
(roughly 1/3 HREL, 1/3 MREL, 1/3 LREL). Do NOT default to HREL.
Look carefully for LREL cues like sensational language or claims unsupported by the article.

Categories:
- HREL: headline is a faithful summary of the article's main event.
- MREL: headline is loosely related, covers a side event or weak inference.
- LREL: headline is misleading, sensational, clickbai ...


## 4. Sarvam wrapper

In [ ]:
LABEL_RE = re.compile(r"\b(HREL|MREL|LREL)\b", re.IGNORECASE)

def parse_label(text):
    if not text:
        return "MREL"
    try:
        obj = json.loads(text.strip())
        if isinstance(obj, dict) and "label" in obj:
            m = LABEL_RE.search(str(obj["label"]))
            if m:
                return m.group(1).upper()
    except Exception:
        pass
    m = LABEL_RE.search(text)
    if m:
        return m.group(1).upper()
    return "MREL"

def call_sarvam(prompt):
    resp = requests.post(
        SARVAM_URL,
        headers={
            "api-subscription-key": SARVAM_KEY,
            "Content-Type": "application/json",
        },
        json={
            "model": "sarvam-105b-conversations",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0,
            "max_tokens": 200,
        },
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"] or ""

def safe_call(fn, prompt, tries=5):
    for i in range(tries):
        try:
            return fn(prompt)
        except Exception as e:
            if i == tries - 1:
                return f"ERROR: {e}"
            wait = 4 * (2 ** i) + random.random()
            print(f"  retry {i+1}: {type(e).__name__}, waiting {wait:.1f}s")
            time.sleep(wait)

MODELS = {
    "sarvam-105b-conversations": (call_sarvam, 2.0),
}

## 5. Runner

In [ ]:
OUT = Path("llm_results_sarvam")
OUT.mkdir(exist_ok=True)

def run(model, prompt, df, n=None):
    n = n or len(df)
    path = OUT / f"{model}__{prompt}__n{n}.csv"
    if path.exists():
        cached = pd.read_csv(path)
        if len(cached) >= n:
            print(f"  using cache: {path.name}")
            return cached

    fn, delay = MODELS[model]
    pfn = PROMPTS[prompt]
    sub = df.head(n)

    rows = []
    for i, r in tqdm(sub.iterrows(), total=len(sub), desc=f"{model}/{prompt}"):
        text = safe_call(fn, pfn(r["article"], r["headline"]))
        rows.append({
            "true": r["label"],
            "pred": parse_label(text),
            "raw": (text or "")[:500],
        })
        if (i + 1) % 50 == 0:
            pd.DataFrame(rows).to_csv(path, index=False)
        time.sleep(delay)

    out = pd.DataFrame(rows)
    out.to_csv(path, index=False)
    return out

## 6. Smoke test

In [ ]:
smoke = run("sarvam-105b-conversations", "zeroshot", test_df, n=50)
print("accuracy:", (smoke["true"] == smoke["pred"]).mean())
print(smoke["pred"].value_counts())
smoke.head()

sarvam-105b-conversations/zeroshot:   0%|          | 0/50 [00:00<?, ?it/s]

accuracy: 0.46
pred
HREL    35
MREL    13
LREL     2
Name: count, dtype: int64


,true,pred,raw
0,LREL,MREL,MREL
1,MREL,HREL,HREL
2,LREL,HREL,HREL
3,HREL,HREL,HREL
4,LREL,LREL,"LREL\n\nReasoning: The headline ""రాజీవ్ను రక్క..."


## 7. Full experiment

400 stratified samples. Uses the same seed as the Groq notebook so comparisons are apples to apples.

In [8]:
eval_df = test_df.reset_index(drop=True)
print(f"running on {len(eval_df)} samples (full test set):")
print(eval_df["label"].value_counts())

print(f"running on {len(eval_df)} samples:")
print(eval_df["label"].value_counts())

results = {}
for model in MODELS:
    for prompt in PROMPTS:
        print(f"\n[ {model} / {prompt} ]")
        results[(model, prompt)] = run(model, prompt, eval_df, n=len(eval_df))

print("\ndone.")

running on 3927 samples (full test set):
label
MREL    1523
HREL    1278
LREL    1126
Name: count, dtype: int64
running on 3927 samples:
label
MREL    1523
HREL    1278
LREL    1126
Name: count, dtype: int64

[ sarvam-105b-conversations / zeroshot ]


sarvam-105b-conversations/zeroshot:   0%|          | 0/3927 [00:00<?, ?it/s]


done.


## 8. Scores

In [9]:
def score(df):
    yt = df["true"].map(lab2id).values
    yp = df["pred"].map(lab2id).fillna(1).astype(int).values
    f1_each = f1_score(yt, yp, labels=[0, 1, 2], average=None)
    return {
        "HREL": round(f1_each[0], 3),
        "MREL": round(f1_each[1], 3),
        "LREL": round(f1_each[2], 3),
        "W": round(f1_score(yt, yp, average="weighted"), 3),
        "M": round(f1_score(yt, yp, average="macro"), 3),
    }

rows = []
for (model, prompt), df in results.items():
    rows.append({"model": model, "prompt": prompt, **score(df)})

llm_scores = pd.DataFrame(rows)
print(llm_scores.to_string(index=False))

                    model   prompt  HREL  MREL  LREL    W     M
sarvam-105b-conversations zeroshot 0.563 0.273 0.211 0.35 0.349


## 9. Comparison with ML baseline and paper BERT

In [10]:
ml_base   = {"HREL": 0.61, "MREL": 0.52, "LREL": 0.60, "W": 0.57, "M": 0.58}
bert_best = {"HREL": 0.68, "MREL": 0.57, "LREL": 0.64, "W": 0.62, "M": 0.63}

line = "=" * 86
print(line)
print("COMPARISON: HREL / MREL / LREL / Weighted / Macro F1")
print(line)
print(f"{'method':<40} {'HREL':>6} {'MREL':>6} {'LREL':>6} {'W':>6} {'M':>6}")
print("-" * 86)

def pr(name, d):
    print(f"{name:<40} {d['HREL']:>6.2f} {d['MREL']:>6.2f} {d['LREL']:>6.2f} {d['W']:>6.2f} {d['M']:>6.2f}")

pr("ML baseline (LR, 5 features)", ml_base)
pr("Paper BERT (mDeBERTa)", bert_best)
print("-" * 86)
for _, r in llm_scores.iterrows():
    pr(f"Zero-shot: {r['model']}", r)
print(line)

COMPARISON: HREL / MREL / LREL / Weighted / Macro F1
method                                     HREL   MREL   LREL      W      M
--------------------------------------------------------------------------------------
ML baseline (LR, 5 features)               0.61   0.52   0.60   0.57   0.58
Paper BERT (mDeBERTa)                      0.68   0.57   0.64   0.62   0.63
--------------------------------------------------------------------------------------
Zero-shot: sarvam-105b-conversations       0.56   0.27   0.21   0.35   0.35


## 10. Confusion matrix

In [11]:
for (model, prompt), df in results.items():
    yt = df["true"].map(lab2id).values
    yp = df["pred"].map(lab2id).fillna(1).astype(int).values
    cm = confusion_matrix(yt, yp, labels=[0, 1, 2])
    print(f"\n[ {model} / {prompt} ]")
    print("           pred HREL  pred MREL  pred LREL")
    for i, lab in enumerate(LABELS):
        print(f"true {lab}:  {cm[i,0]:>8}   {cm[i,1]:>8}   {cm[i,2]:>8}")


[ sarvam-105b-conversations / zeroshot ]
           pred HREL  pred MREL  pred LREL
true HREL:      1206         58         14
true MREL:      1172        307         44
true LREL:       626        360        140
